# Train Fake News Detection on Colab
Notebook nay clone repo, cai dependencies, train model (co/khong curriculum), va luu artifacts.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Ban nen chuyen Runtime sang GPU de train nhanh hon.')

In [ ]:
# ===== Cau hinh repo =====
REPO_URL = 'https://github.com/<username>/<repo>.git'  # doi thanh repo cua ban
BRANCH = 'main'
REPO_DIR = '/content/fake-new-detection'

# Neu repo private: dat USE_PRIVATE_REPO = True va dien token
USE_PRIVATE_REPO = False
GITHUB_TOKEN = ''  # ghp_xxx

# ===== Cau hinh train =====
USE_CURRICULUM = True
EPOCHS_OVERRIDE = 5
BATCH_SIZE_OVERRIDE = 16

In [ ]:
import os
import shutil
import subprocess

if USE_PRIVATE_REPO:
    if not GITHUB_TOKEN:
        raise ValueError('Repo private thi can GITHUB_TOKEN')
    clone_url = REPO_URL.replace('https://', f'https://{GITHUB_TOKEN}@')
else:
    clone_url = REPO_URL

if os.path.exists(REPO_DIR):
    print('Repo da ton tai, pull code moi...')
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)
else:
    print('Dang clone repo...')
    subprocess.run(['git', 'clone', '-b', BRANCH, clone_url, REPO_DIR], check=True)

print('San sang tai:', REPO_DIR)

In [ ]:
%cd /content/fake-new-detection

# Cai dependencies cho module model
!pip -q install -r requirements-model.txt

# pandas read_parquet can pyarrow
!pip -q install pyarrow

In [ ]:
from pathlib import Path

required_files = [
    Path('data/datasets/normalized/train.parquet'),
    Path('data/datasets/normalized/val.parquet'),
    Path('data/datasets/normalized/test.parquet'),
]

missing = [str(p) for p in required_files if not p.exists()]
if missing:
    print('Thieu dataset normalized, tao mock data de train thu:')
    for p in missing:
        print(' -', p)
    !python data/create_mock_data.py
else:
    print('Dataset normalized day du, bo qua buoc tao mock data.')

In [ ]:
import shlex
import subprocess

cmd = ['python', 'model/train.py', '--config', 'model/config.yaml']
if USE_CURRICULUM:
    cmd.append('--curriculum')
if EPOCHS_OVERRIDE is not None:
    cmd.extend(['--epochs', str(EPOCHS_OVERRIDE)])
if BATCH_SIZE_OVERRIDE is not None:
    cmd.extend(['--batch-size', str(BATCH_SIZE_OVERRIDE)])

print('Run command:', ' '.join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, check=True)

In [ ]:
from pathlib import Path

artifacts = [
    Path('models/final_model.pt'),
    Path('models/lora_adapter'),
    Path('models/checkpoints'),
]
for p in artifacts:
    print(f'{p}:', 'OK' if p.exists() else 'MISSING')

print('\nTop checkpoint files:')
for ckpt in sorted(Path('models/checkpoints').glob('*.ckpt'))[:10]:
    print(' -', ckpt)

In [ ]:
# Optional: luu artifact sang Google Drive
SAVE_TO_DRIVE = False
DRIVE_TARGET = '/content/drive/MyDrive/fake-news-models'

if SAVE_TO_DRIVE:
    from google.colab import drive
    import shutil
    import os

    drive.mount('/content/drive')
    os.makedirs(DRIVE_TARGET, exist_ok=True)
    shutil.copytree('models', f'{DRIVE_TARGET}/models', dirs_exist_ok=True)
    print('Da copy models ->', DRIVE_TARGET)
else:
    print('Dat SAVE_TO_DRIVE = True neu muon backup len Google Drive.')

In [ ]:
# Optional: zip artifact de download
!zip -rq models_artifacts.zip models
print('Created models_artifacts.zip')
# from google.colab import files
# files.download('models_artifacts.zip')